# Analytic Solov'ev equilibrium

Construct a constant-source Grad–Shafranov solution from physical boundary
constraints, compare analytic and discretized fields, and export it to VAFT's
lightweight in-memory representation.

The example reproduces the classic nested Solov'ev solution
$\psi = R^2 Z^2/\kappa^2 + (R^2-R_0^2)^2/4$ through the constraint solver:
its flux surfaces are closed and nested by construction, so the export's
boundary-closure validation is guaranteed to succeed and the magnetic axis is
located automatically. `evaluate_solovev` works with $\psi$ in Wb/rad; the
export honors the declared COCOS, so requesting the (default) full-weber
convention 11 stores $2\pi\psi$.


In [ ]:
import numpy as np
from vaft.data.equilibrium import SolovevConstraint
from vaft.formula.constants import MU0
from vaft.process.equilibrium import evaluate_solovev, solve_solovev_constraints, solovev_to_equilibrium

R0, kappa, psi_b = 1.0, 1.4, 0.08
pprime = -8.0 * ((1.0 + 1.0 / kappa**2) / 4.0) / MU0  # supplies the R^4 term
r_out = np.sqrt(R0**2 + 2 * np.sqrt(psi_b))
r_in = np.sqrt(R0**2 - 2 * np.sqrt(psi_b))
z_top = kappa * np.sqrt(psi_b) / R0
constraints = [
    SolovevConstraint(r_out, 0.0, "psi", psi_b),
    SolovevConstraint(r_in, 0.0, "psi", psi_b),
    SolovevConstraint(R0, z_top, "psi", psi_b),
    SolovevConstraint(R0, 0.0, "psi", 0.0),
    SolovevConstraint(R0, 0.0, "dpsi_dr", 0.0),
]
model = solve_solovev_constraints(
    constraints, pprime=pprime, ffprime=0.0, rref=R0, psi_boundary=psi_b, f_boundary=1.4
)
print("rank/residual:", model.rank, model.residual_norm)
r = np.linspace(0.5, 1.5, 151); z = np.linspace(-0.7, 0.7, 151)
rm, zm = np.meshgrid(r, z, indexing="ij")
analytic = evaluate_solovev(model, rm, zm)
# The magnetic axis is located automatically (O-point with a closed boundary).
equilibrium = solovev_to_equilibrium(model, r, z, convention=11)
print("portable grid:", equilibrium.psi.shape, "LCFS points:", equilibrium.lcfs.r.size)
print("magnetic axis:", equilibrium.magnetic_axis, "LCFS closed:", equilibrium.lcfs.closed)


In [ ]:
# COCOS 11 stores the full poloidal flux (2*pi*psi); divide the gradient by
# 2*pi to recover the per-radian fields evaluate_solovev works with.
dpsi_dr = np.gradient(equilibrium.psi, r, axis=0, edge_order=2) / (2 * np.pi)
dpsi_dz = np.gradient(equilibrium.psi, z, axis=1, edge_order=2) / (2 * np.pi)
br_grid = -dpsi_dz/rm; bz_grid = dpsi_dr/rm
np.testing.assert_allclose(br_grid[3:-3, 3:-3], analytic["b_r"][3:-3, 3:-3], rtol=1e-2, atol=3e-2)
np.testing.assert_allclose(bz_grid[3:-3, 3:-3], analytic["b_z"][3:-3, 3:-3], rtol=1e-2, atol=3e-2)
print("Analytic and discretized poloidal fields agree within the grid tolerance.")
